# Task 1 — QLoRA trên gold có compile log

Mặc định notebook chỉ huấn luyện trên **26 bài gold train / 6 bài gold validation**, lấy từ [sample_dataset_with_compile_logs.zip](https://drive.google.com/drive/folders/1HOad-pMh1Kqlgu5sutUg39jWW5OYuo9Q). Đặt `USE_SILVER = True` nếu muốn chạy lại đối chứng silver→gold; khi đó public và silver được tải từ [thư mục Drive cũ](https://drive.google.com/drive/folders/1kBd4pE4JHcDtm0WFYueFq6aTEuTanUNC). Mỗi run tải ZIP mới, ghi SHA-256 và lưu vào thư mục output riêng. Chỉ đánh giá và chọn adapter bằng **gold validation**. Chọn runtime **GPU** trong Colab.

Quy trình: cài thư viện → clone mới branch `task-1-tien` → tải và chuẩn bị gold → preflight token → smoke test (một bước gold và suy luận) → train gold 3 epoch. Push code của branch `task-1-tien` trước khi chạy notebook vì Colab clone code từ GitHub. ZIP cần quyền xem qua gdown.

Các output đang hiển thị trong file notebook là kết quả chạy trước đó được giữ lại; hãy **Run all** để tạo báo cáo gold mới.


In [1]:
%pip -q install "transformers>=4.46,<5" "peft>=0.13,<1" "accelerate>=0.34,<2" "bitsandbytes>=0.45,<1" "pyyaml>=6" "pydantic>=2.5" gdown

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 141.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 60.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > GPU rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))

PROJECT_HOME = Path('/content/drive/MyDrive/LLM/Final Project')
REPO_URL = 'https://github.com/tiendinhquang2104/llm-final-project.git'
REPO_BRANCH = 'task-1-tien'
from datetime import datetime, timezone
# Clone fresh on each run so an older Colab checkout with local edits is preserved.
REPO_DIR = Path('/content') / ('llm-final-project-task1-finetune-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
                '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
required_source = ['scripts/train_task1.py', 'src/task1/finetune.py', 'src/task1/finetune_public.py', 'configs/task1/lora_7b.yaml', 'scripts/prepare_data.py', 'scripts/prepare_task1_silver.py', 'src/task1/silver_data.py']
missing_source = [name for name in required_source if not (REPO_DIR / name).is_file()]
if missing_source:
    raise FileNotFoundError(f'Branch {REPO_BRANCH} trên GitHub chưa có code fine-tune: {missing_source}. Push branch rồi chạy lại.')
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
sys.path = [str(REPO_DIR)] + [p for p in sys.path if p != str(REPO_DIR)]
for module in list(sys.modules):
    if module == 'src' or module.startswith('src.'):
        del sys.modules[module]
print('Source:', REPO_DIR, '| branch:', REPO_BRANCH, '| commit:', commit)

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
Source: /content/llm-final-project-task1-finetune-20261019T084700133528Z | branch: task-1-tien | commit: 0b24b30


In [3]:
import gdown, hashlib, shutil, zipfile
from datetime import datetime, timezone
from src.data.loader import resolve_dataset_root

USE_SILVER = False  # Mặc định chỉ huấn luyện trên gold; True để đối chứng silver→gold.
GOLD_FOLDER_URL = 'https://drive.google.com/drive/folders/1HOad-pMh1Kqlgu5sutUg39jWW5OYuo9Q'
AUX_FOLDER_URL = 'https://drive.google.com/drive/folders/1kBd4pE4JHcDtm0WFYueFq6aTEuTanUNC'
DRIVE_FILES = {
    'gold': {'id': '1hCjmWNSPQ9qTsox7RakbFzLWsW7qvfwy', 'name': 'sample_dataset_with_compile_logs.zip', 'folder': GOLD_FOLDER_URL},
    'gold_public': {'id': '1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj', 'name': 'public-test-cham-diem-phan-hoi-files.zip', 'folder': AUX_FOLDER_URL},
    'silver': {'id': '1oESzEa-vwppDD-dDTmAVk7KIHqUsgowl', 'name': 'generate-dataset.zip', 'folder': AUX_FOLDER_URL},
}
active_files = DRIVE_FILES if USE_SILVER else {'gold': DRIVE_FILES['gold']}
RUN_TIME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_KIND = 'silver-gold' if USE_SILVER else 'gold'
DOWNLOAD_DIR = Path('/content') / f'task1-{RUN_KIND}-{RUN_TIME}'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=False)

# Always download into this run's empty directory; never reuse a same-name ZIP.
archives = {}
source_manifest = {'drive_folder': GOLD_FOLDER_URL, 'aux_drive_folder': AUX_FOLDER_URL if USE_SILVER else None, 'use_silver': USE_SILVER, 'downloaded_at_utc': RUN_TIME, 'git_commit': commit, 'files': {}}
for key, spec in active_files.items():
    path = DOWNLOAD_DIR / spec['name']
    downloaded = gdown.download(id=spec['id'], output=str(path), quiet=False)
    if not downloaded or not path.is_file() or not zipfile.is_zipfile(path):
        raise RuntimeError(f'Không tải được ZIP hợp lệ: {spec["name"]}')
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    with zipfile.ZipFile(path) as zf:
        bad = zf.testzip()
        if bad:
            raise ValueError(f'{spec["name"]} lỗi CRC ở {bad}')
    archives[key] = path
    source_manifest['files'][key] = {
        'drive_file_id': spec['id'], 'drive_folder': spec['folder'], 'name': spec['name'],
        'sha256': digest, 'bytes': path.stat().st_size,
    }
    print(spec['name'], 'SHA-256:', digest)

def extract_safe(archive, destination):
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(archive) as zf:
        root = destination.resolve()
        for info in zf.infolist():
            target = (root / info.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'ZIP chứa đường dẫn không an toàn: {info.filename}')
        zf.extractall(destination)

def extract_nested(outer_path, inner_name, destination):
    with zipfile.ZipFile(outer_path) as outer:
        matches = [name for name in outer.namelist() if Path(name).name == inner_name]
        if len(matches) != 1:
            raise FileNotFoundError(f'ZIP ngoài phải có đúng một {inner_name}; thấy {matches}')
        inner = DOWNLOAD_DIR / inner_name
        with outer.open(matches[0]) as src, inner.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    if not zipfile.is_zipfile(inner):
        raise ValueError(f'ZIP lồng không hợp lệ: {inner_name}')
    source_manifest['files'][inner_name] = {
        'parent': 'gold_public', 'sha256': hashlib.sha256(inner.read_bytes()).hexdigest(),
        'bytes': inner.stat().st_size,
    }
    extract_safe(inner, destination)
    return resolve_dataset_root(destination)

extract_safe(archives['gold'], DOWNLOAD_DIR / 'gold')
GOLD_ROOT = resolve_dataset_root(DOWNLOAD_DIR / 'gold')
gold_samples = json.loads((GOLD_ROOT / 'task1_grading.json').read_text(encoding='utf-8'))['samples']
assert len(gold_samples) == 32 and sum(bool((row['input'].get('compile_log') or '').strip()) for row in gold_samples) >= 6, 'Gold ZIP phải có 32 bài và compile log cho các bài có chẩn đoán'
if USE_SILVER:
    PUBLIC_ROOT = extract_nested(archives['gold_public'], 'public_test_release.zip', DOWNLOAD_DIR / 'public')
    extract_safe(archives['silver'], DOWNLOAD_DIR / 'silver')
    SILVER_ROOT = resolve_dataset_root(DOWNLOAD_DIR / 'silver')
RUN_ID = RUN_TIME + '-' + source_manifest['files']['gold']['sha256'][:8] + ('-' + source_manifest['files']['silver']['sha256'][:8] if USE_SILVER else '')
DATA_HOME = PROJECT_HOME / ('data/task1_finetune_silver_gold' if USE_SILVER else 'data/task1_finetune_gold') / RUN_ID
PROCESSED_DIR = DATA_HOME / 'gold_processed'
SPLITS_DIR = DATA_HOME / 'gold_splits'
SILVER_PROCESSED_DIR = DATA_HOME / 'silver_processed'
OUTPUT_DIR = PROJECT_HOME / ('data/outputs/task1/qwen25_coder7b_qlora_silver_gold' if USE_SILVER else 'data/outputs/task1/qwen25_coder7b_qlora_gold') / RUN_ID
DATA_HOME.mkdir(parents=True, exist_ok=False)
SOURCE_MANIFEST = DATA_HOME / 'source_manifest.json'
SOURCE_MANIFEST.write_text(json.dumps(source_manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
subprocess.run([sys.executable, str(REPO_DIR / 'scripts/prepare_data.py'),
                '--data_dir', str(GOLD_ROOT), '--output_dir', str(PROCESSED_DIR),
                '--splits_dir', str(SPLITS_DIR), '--seed', '42', '--val_ratio', '0.2'],
               cwd=REPO_DIR, check=True)
if USE_SILVER:
    subprocess.run([sys.executable, str(REPO_DIR / 'scripts/prepare_task1_silver.py'),
                    '--silver-root', str(SILVER_ROOT), '--gold-root', str(GOLD_ROOT),
                    '--public-root', str(PUBLIC_ROOT), '--output-dir', str(SILVER_PROCESSED_DIR)],
                   cwd=REPO_DIR, check=True)
gold_manifest = json.loads((PROCESSED_DIR / 'manifest.json').read_text(encoding='utf-8'))
assert (gold_manifest['n_samples'], gold_manifest['n_train_ids'], gold_manifest['n_val_ids']) == (32, 26, 6), gold_manifest
if USE_SILVER:
    silver_manifest = json.loads((SILVER_PROCESSED_DIR / 'silver_manifest.json').read_text(encoding='utf-8'))
    assert (silver_manifest['n_source_task1'], silver_manifest['n_accepted'], silver_manifest['n_excluded']) == (462, 422, 40), silver_manifest
    assert (silver_manifest['n_heldout_not_read'], silver_manifest['n_task3_only_not_read']) == (65, 150), silver_manifest
for part in ('train', 'val'):
    fixed_ids = json.loads((REPO_DIR / 'data/splits' / f'{part}_ids.json').read_text(encoding='utf-8'))
    current_ids = json.loads((SPLITS_DIR / f'{part}_ids.json').read_text(encoding='utf-8'))
    assert fixed_ids == current_ids, f'Gold {part} split IDs đã thay đổi'
print('Gold train/val:', 26, '/', 6, '| Silver enabled:', USE_SILVER)
if USE_SILVER:
    print('Silver accepted:', silver_manifest['n_accepted'], '| Audit:', SILVER_PROCESSED_DIR / 'silver_manifest.json')
print('Output:', OUTPUT_DIR)


Downloading...
From: https://drive.google.com/uc?id=1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj
To: /content/task1-silver-gold-20261019T084702143180Z/public-test-cham-diem-phan-hoi-files.zip
100%|██████████| 274k/274k [00:00<00:00, 86.1MB/s]


public-test-cham-diem-phan-hoi-files.zip SHA-256: 637a033ead19dc2fd5c5a82c859e92637762b09f4da870c2d012f108a9183dad


Downloading...
From: https://drive.google.com/uc?id=1oESzEa-vwppDD-dDTmAVk7KIHqUsgowl
To: /content/task1-silver-gold-20261019T084702143180Z/generate-dataset.zip
100%|██████████| 566k/566k [00:00<00:00, 106MB/s]


generate-dataset.zip SHA-256: 88eff95f2841efd061ab0203907f9142ad6a2a2b72e266a4d55043a50aedac9c
Gold train/val: 26 / 6 | Silver accepted: 422
Filtered silver audit: /content/drive/MyDrive/LLM/Final Project/data/task1_finetune_silver_gold/20261019T084702143180Z-637a033e-88eff95f/silver_processed/silver_manifest.json
Output: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_silver_gold/20261019T084702143180Z-637a033e-88eff95f


In [4]:
RUNNER = REPO_DIR / 'scripts/train_task1.py'
CONFIG = REPO_DIR / 'configs/task1/lora_7b.yaml'

def run_task1(*flags):
    command = [sys.executable, str(RUNNER), '--config', str(CONFIG),
               '--processed-dir', str(PROCESSED_DIR), '--splits-dir', str(SPLITS_DIR),
               '--source-manifest', str(SOURCE_MANIFEST),
               '--output-dir', str(OUTPUT_DIR)]
    if USE_SILVER:
        command += ['--silver-processed-dir', str(SILVER_PROCESSED_DIR), '--stage', 'all']
    command += list(flags)
    subprocess.run(command, cwd=REPO_DIR, check=True)

run_task1('--preflight')


## Smoke test và huấn luyện đầy đủ

Mặc định smoke test chạy **một bước gold** rồi sinh dự đoán cho một bài gold validation. Lượt đầy đủ train 26 gold trong 3 epoch và chọn epoch theo QWK, hòa thì MAE, trên đúng 6 gold validation. Khi bật `USE_SILVER`, notebook chạy thêm 1 epoch silver → 3 epoch gold và chọn giữa hai nhánh bằng gold validation. Mỗi lần chạy có output riêng.


In [5]:
RUN_SMOKE = True
if RUN_SMOKE:
    run_task1('--smoke')
    smoke_summary = json.loads((OUTPUT_DIR / 'smoke/training_summary.json').read_text(encoding='utf-8'))
    print('Smoke test OK. Selected:', smoke_summary['selected_variant'])
    print('Gold validation QWK:', smoke_summary['selected_metrics']['qwk'])


Smoke test OK. Selected: gold_only
Gold validation QWK: 0.0


In [6]:
RUN_FULL = True
if RUN_FULL:
    run_task1()
    summary = json.loads((OUTPUT_DIR / 'training_summary.json').read_text(encoding='utf-8'))
    print('Selected variant:', summary['selected_variant'], '| epoch:', summary['selected_epoch'])
    print('Gold-only QWK/MAE:', summary['gold_only_best']['metrics']['qwk'], summary['gold_only_best']['metrics']['mae'])
    if summary['silver_then_gold_best']:
        print('Silver→gold QWK/MAE:', summary['silver_then_gold_best']['metrics']['qwk'], summary['silver_then_gold_best']['metrics']['mae'])
    print('Selected valid output rate:', summary['selected_metrics']['valid_output_rate'])
    print('Adapter:', OUTPUT_DIR / 'best_adapter')
    print('Report:', OUTPUT_DIR / 'gold_only/validation_report.json')
    if summary['silver_then_gold_best']:
        print('Silver report:', OUTPUT_DIR / 'silver_then_gold/validation_report.json')
    print('Source hashes:', OUTPUT_DIR / 'source_manifest.json')


Selected variant: gold_only | epoch: 2
Gold-only QWK/MAE: 0.29025844930417477 3.5
Silver→gold QWK/MAE: 0.19607843137254888 3.8333333333333335
Selected valid output rate: 0.8333333333333334
Adapter: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_silver_gold/20261019T084702143180Z-637a033e-88eff95f/best_adapter
Reports: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_silver_gold/20261019T084702143180Z-637a033e-88eff95f/gold_only/validation_report.json /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_silver_gold/20261019T084702143180Z-637a033e-88eff95f/silver_then_gold/validation_report.json
Source hashes: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_silver_gold/20261019T084702143180Z-637a033e-88eff95f/source_manifest.json
